# Chapter 7: Numerical Methods & Floating Point Stability

**Mathematics Behind LLMs — Book Series**

This chapter examines the numerical methods that keep large language model training and inference numerically sound.
We cover IEEE 754 floating-point formats, catastrophic cancellation, condition numbers, numerically stable
softmax and LayerNorm, online softmax for Flash Attention, INT8 quantization, Kahan summation, and
mixed-precision training best practices.


## 7.0 Setup


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")


## 7.1 IEEE 754 Floating-Point Formats

Modern deep learning relies on three floating-point formats:

| Format | Sign | Exponent | Mantissa | Total bits |
|--------|------|----------|----------|------------|
| FP32   | 1    | 8        | 23       | 32         |
| FP16   | 1    | 5        | 10       | 16         |
| BF16   | 1    | 8        | 7        | 16         |

The stored value of a normal FP32 number is:

$$v = (-1)^{s} \cdot 2^{e - \text{bias}} \cdot \left(1 + \frac{m}{2^{23}}\right)$$

where $s$ is the sign bit, $e$ is the biased exponent (bias = 127 for FP32), and $m$ is the mantissa integer.

**Machine epsilon** for FP32 is approximately $\epsilon_{\text{mach}} \approx 1.2 \times 10^{-7}$,
meaning that any value smaller than $\epsilon_{\text{mach}}$ relative to 1.0 is indistinguishable from zero.

BF16 keeps the same 8-bit exponent as FP32 (same dynamic range), but only 7 mantissa bits (lower precision).
FP16 has a much smaller exponent range — it overflows at 65504 — which is why gradients can underflow or overflow
during mixed-precision training without loss scaling.


In [ ]:
# Inspect floating-point format properties via torch.finfo
for dtype in [torch.float32, torch.float16, torch.bfloat16]:
    info = torch.finfo(dtype)
    bits = torch.tensor(0, dtype=dtype).element_size() * 8
    print(f"{str(dtype):<20}  bits={bits:2d}  eps={info.eps:.3e}  "
          f"max={info.max:.3e}  tiny={info.tiny:.3e}")

print()

# Memory footprint for a 1B-parameter model
n_params = 1_000_000_000
for dtype, label in [(torch.float32, 'FP32'), (torch.float16, 'FP16'), (torch.bfloat16, 'BF16')]:
    bytes_per_param = torch.tensor(0, dtype=dtype).element_size()
    gb = n_params * bytes_per_param / 1e9
    print(f"{label}: {gb:.1f} GB for 1B params")


## 7.2 Catastrophic Cancellation

**Catastrophic cancellation** occurs when two nearly equal floating-point numbers are subtracted,
causing most of the significant bits to cancel and leaving only the low-order bits — which are dominated
by rounding error — in the result.

A classic example is computing the **sample variance** via the one-pass formula:

$$\text{Var}(X) = E[X^2] - (E[X])^2$$

When the mean is large compared with the spread, both terms are nearly equal and their difference loses
many significant digits.  The **Welford online algorithm** avoids this by accumulating the
mean and sum-of-squared-deviations simultaneously:

$$\delta_n = x_n - \mu_{n-1}, \quad \mu_n = \mu_{n-1} + \delta_n / n, \quad M_n = M_{n-1} + \delta_n(x_n - \mu_n)$$

$$\text{Var} = M_n / (n-1) \quad \text{(sample variance)}$$


In [ ]:
# --- Catastrophic Cancellation Demo: (a + b) - a ---
a = torch.tensor(1e4, dtype=torch.float32)
b = torch.tensor(1e-4, dtype=torch.float32)
result_f32 = (a + b) - a
print(f"FP32  (a+b)-a = {result_f32.item():.6e}  (expected {b.item():.6e})")

a16 = a.to(torch.float16)
b16 = b.to(torch.float16)
result_f16 = (a16 + b16) - a16
print(f"FP16  (a+b)-a = {result_f16.item():.6e}  (expected {b16.item():.6e})")

print()

# --- Naive vs Welford variance ---
def naive_variance(x: torch.Tensor) -> torch.Tensor:
    """Unstable one-pass formula: E[X^2] - (E[X])^2."""
    return x.pow(2).mean() - x.mean().pow(2)

def welford_variance(x: torch.Tensor) -> torch.Tensor:
    """Numerically stable Welford online algorithm."""
    n = 0
    mean = torch.tensor(0.0)
    M2 = torch.tensor(0.0)
    for xi in x:
        n += 1
        delta = xi - mean
        mean = mean + delta / n
        delta2 = xi - mean
        M2 = M2 + delta * delta2
    return M2 / (n - 1)

# Data with large mean, small variance — worst case for naive
torch.manual_seed(0)
data = torch.randn(1000) * 0.01 + 1e6  # mean ~1e6, std ~0.01
true_var = data.to(torch.float64).var().item()

print(f"True variance (FP64) : {true_var:.6e}")
print(f"Naive variance (FP32): {naive_variance(data).item():.6e}")
print(f"Welford variance     : {welford_variance(data).item():.6e}")
print(f"torch.var            : {data.var().item():.6e}")


## 7.3 Condition Numbers

The **condition number** of a matrix $\mathbf{A}$ measures how sensitive the solution of $\mathbf{A}\mathbf{x}=\mathbf{b}$
is to perturbations in $\mathbf{b}$ or $\mathbf{A}$:

$$\kappa(\mathbf{A}) = \|\mathbf{A}\| \cdot \|\mathbf{A}^{-1}\| = \frac{\sigma_{\max}}{\sigma_{\min}}$$

where $\sigma_{\max}$ and $\sigma_{\min}$ are the largest and smallest singular values.

A relative error bound for the solution is:

$$\frac{\|\delta \mathbf{x}\|}{\|\mathbf{x}\|} \leq \kappa(\mathbf{A}) \cdot \frac{\|\delta \mathbf{b}\|}{\|\mathbf{b}\|}$$

If $\kappa(\mathbf{A}) \approx 10^6$ and we have FP32 precision ($\epsilon \approx 10^{-7}$),
we lose roughly 6 digits of accuracy — leaving us with almost no reliable digits.

In transformer attention, large query-key dot products can produce ill-conditioned softmax Jacobians,
motivating the $1/\sqrt{d_k}$ scaling factor.


In [ ]:
torch.manual_seed(42)

# Well-conditioned matrix
A_good = torch.eye(4) + 0.1 * torch.randn(4, 4)
kappa_good = torch.linalg.cond(A_good).item()

# Ill-conditioned matrix: nearly singular (rows almost linearly dependent)
v = torch.randn(4)
A_bad = torch.outer(v, v) + 1e-4 * torch.eye(4)
kappa_bad = torch.linalg.cond(A_bad).item()

print(f"Well-conditioned kappa : {kappa_good:.2f}")
print(f"Ill-conditioned kappa  : {kappa_bad:.2e}")

# Solve Ax = b with a small perturbation to b
b = torch.randn(4)
delta_b = 1e-4 * torch.randn(4)

for label, A, kappa in [("well-cond", A_good, kappa_good), ("ill-cond", A_bad, kappa_bad)]:
    x = torch.linalg.solve(A, b)
    x_perturbed = torch.linalg.solve(A, b + delta_b)
    rel_input_change = (delta_b.norm() / b.norm()).item()
    rel_output_change = ((x_perturbed - x).norm() / x.norm()).item()
    amplification = rel_output_change / rel_input_change
    print(f"[{label}] kappa={kappa:.2e}  rel_input_change={rel_input_change:.2e}  "
          f"rel_output_change={rel_output_change:.2e}  amplification={amplification:.2f}")


## 7.4 Numerically Stable Softmax

The **naive softmax** computes:

$$\text{softmax}(z_i) = \frac{e^{z_i}}{\sum_j e^{z_j}}$$

For $z_i > 88$ in FP32, $e^{z_i}$ overflows to $+\infty$, and for $z_i < -104$ it underflows to 0,
producing NaN or all-zeros output.

The **numerically stable softmax** subtracts the maximum before exponentiating:

$$\text{softmax}(z_i) = \frac{e^{z_i - m}}{\sum_j e^{z_j - m}}, \quad m = \max_j z_j$$

This is mathematically equivalent (the constant $e^{-m}$ cancels in numerator and denominator)
but numerically safe because all exponents are $\leq 0$.

**Log-softmax** combines the log and softmax to avoid computing intermediate $e^{z_i}$:

$$\log \text{softmax}(z_i) = z_i - m - \log \sum_j e^{z_j - m}$$


In [ ]:
def naive_softmax(z: torch.Tensor) -> torch.Tensor:
    exp_z = torch.exp(z)
    return exp_z / exp_z.sum()

def stable_softmax(z: torch.Tensor) -> torch.Tensor:
    m = z.max()
    exp_z = torch.exp(z - m)
    return exp_z / exp_z.sum()

def stable_log_softmax(z: torch.Tensor) -> torch.Tensor:
    m = z.max()
    log_sum_exp = m + torch.log(torch.exp(z - m).sum())
    return z - log_sum_exp

# Normal inputs
z_normal = torch.tensor([1.0, 2.0, 3.0, 4.0])
print("Normal inputs:")
print(f"  naive  : {naive_softmax(z_normal)}")
print(f"  stable : {stable_softmax(z_normal)}")
print(f"  F.softmax: {F.softmax(z_normal, dim=0)}")

print()

# Large inputs that cause overflow
z_large = torch.tensor([100.0, 200.0, 300.0, 400.0])
print("Large inputs (overflow test):")
print(f"  naive  : {naive_softmax(z_large)}   <- NaN due to overflow")
print(f"  stable : {stable_softmax(z_large)}  <- correct")
print(f"  F.softmax: {F.softmax(z_large, dim=0)}")

print()

# Log-softmax comparison
print("Log-softmax:")
print(f"  stable_log_softmax : {stable_log_softmax(z_normal)}")
print(f"  F.log_softmax      : {F.log_softmax(z_normal, dim=0)}")


## 7.5 Numerically Stable LayerNorm

**Layer Normalization** normalizes each sample across its feature dimension:

$$\hat{x}_i = \gamma \cdot \frac{x_i - \mu}{\sqrt{\sigma^2 + \epsilon}} + \beta$$

where $\mu = \frac{1}{d}\sum_i x_i$, $\sigma^2 = \frac{1}{d}\sum_i (x_i - \mu)^2$, and $\epsilon > 0$
(typically $10^{-5}$) prevents division by zero when the input is constant.

**Two-pass algorithm**: first compute $\mu$, then compute $\sigma^2$ using $\mu$.
This is stable but requires two passes over the data.

**Welford online algorithm** (single pass): maintains mean and variance simultaneously:

$$\delta_n = x_n - \mu_{n-1}, \quad \mu_n = \mu_{n-1} + \delta_n/n, \quad M_n = M_{n-1} + \delta_n(x_n - \mu_n)$$

Final variance: $\sigma^2 = M_n / n$.


In [ ]:
class WelfordLayerNorm(nn.Module):
    """LayerNorm using Welford's online algorithm for variance."""

    def __init__(self, normalized_shape: int, eps: float = 1e-5):
        super().__init__()
        self.eps = eps
        self.gamma = nn.Parameter(torch.ones(normalized_shape))
        self.beta = nn.Parameter(torch.zeros(normalized_shape))

    def _welford_mean_var(self, x: torch.Tensor):
        """Compute (mean, variance) via Welford over last dim."""
        d = x.shape[-1]
        mean = torch.zeros(*x.shape[:-1], device=x.device, dtype=x.dtype)
        M2 = torch.zeros_like(mean)
        for n in range(1, d + 1):
            xi = x[..., n - 1]
            delta = xi - mean
            mean = mean + delta / n
            delta2 = xi - mean
            M2 = M2 + delta * delta2
        return mean, M2 / d

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        mean, var = self._welford_mean_var(x)
        x_hat = (x - mean.unsqueeze(-1)) / torch.sqrt(var.unsqueeze(-1) + self.eps)
        return self.gamma * x_hat + self.beta

torch.manual_seed(0)
d = 16
x = torch.randn(4, d)

welford_ln = WelfordLayerNorm(d)
torch_ln = nn.LayerNorm(d, elementwise_affine=True)
# Copy weights so outputs are comparable
torch_ln.weight.data = welford_ln.gamma.data.clone()
torch_ln.bias.data = welford_ln.beta.data.clone()

out_welford = welford_ln(x)
out_torch = torch_ln(x)
print(f"Max absolute difference (normal input): {(out_welford - out_torch).abs().max().item():.2e}")

# Edge case: constant input (variance = 0, eps kicks in)
x_const = torch.full((2, d), 5.0)
out_w = welford_ln(x_const)
out_t = torch_ln(x_const)
print(f"Max absolute difference (constant input): {(out_w - out_t).abs().max().item():.2e}")

# Edge case: single element (d=1)
wln1 = WelfordLayerNorm(1)
tln1 = nn.LayerNorm(1, elementwise_affine=True)
tln1.weight.data = wln1.gamma.data.clone()
tln1.bias.data = wln1.beta.data.clone()
x1 = torch.randn(4, 1)
print(f"Max absolute difference (single element): {(wln1(x1) - tln1(x1)).abs().max().item():.2e}")


## 7.6 Online Softmax for Flash Attention

Flash Attention avoids materializing the full $N \times N$ attention matrix by processing the sequence
in blocks and maintaining running statistics per row.

For each row, maintain two scalars:
- $m$: running maximum of dot products seen so far
- $\ell$: running sum of $e^{x - m}$ values

When a new block arrives with block-level maximum $m_{\text{block}}$ and block-level sum $\ell_{\text{block}}$:

$$m_{\text{new}} = \max(m,\, m_{\text{block}})$$

$$\ell_{\text{new}} = e^{m - m_{\text{new}}} \cdot \ell + e^{m_{\text{block}} - m_{\text{new}}} \cdot \ell_{\text{block}}$$

The output accumulation (weighted sum of values) is similarly rescaled:

$$O_{\text{new}} = \frac{e^{m - m_{\text{new}}} \cdot \ell \cdot O + e^{m_{\text{block}} - m_{\text{new}}} \cdot O_{\text{block}}}{\ell_{\text{new}}}$$

The final result is **exactly** the standard softmax — no approximation, no materializing $N \times N$.


In [ ]:
def online_softmax(x: torch.Tensor, block_size: int = 4) -> torch.Tensor:
    """
    Compute softmax of 1-D tensor x using the online (Flash-Attention-style)
    two-pass algorithm with blocks of size `block_size`.
    Returns the same result as F.softmax(x, dim=0).
    """
    n = x.shape[0]
    m = torch.tensor(float('-inf'))  # running max
    ell = torch.tensor(0.0)          # running normalizer

    # --- Pass 1: accumulate (m, ell) block by block ---
    for start in range(0, n, block_size):
        block = x[start:start + block_size]
        m_block = block.max()
        ell_block = torch.exp(block - m_block).sum()

        m_new = torch.max(m, m_block)
        ell = torch.exp(m - m_new) * ell + torch.exp(m_block - m_new) * ell_block
        m = m_new

    # --- Pass 2: compute final softmax ---
    return torch.exp(x - m) / ell

torch.manual_seed(7)
# Use 12 elements split into 3 blocks of 4
x = torch.randn(12)

out_online = online_softmax(x, block_size=4)
out_ref = F.softmax(x, dim=0)

print("Online softmax  :", out_online.tolist())
print("Reference softmax:", out_ref.tolist())
print(f"Max absolute diff : {(out_online - out_ref).abs().max().item():.2e}")
print(f"Sums to 1.0 (online): {out_online.sum().item():.8f}")
print(f"Sums to 1.0 (ref)   : {out_ref.sum().item():.8f}")

# Test with inputs that would overflow naive softmax
x_large = x * 50  # amplify to create potential overflow
out_large = online_softmax(x_large, block_size=4)
print(f"\nLarge inputs — any NaN: {out_large.isnan().any().item()}")
print(f"Large inputs match F.softmax: "
      f"{(out_large - F.softmax(x_large, dim=0)).abs().max().item():.2e}")


## 7.7 INT8 Quantization

**Post-training quantization** maps FP32 weights to INT8 integers:

$$W_q = \text{round}\!\left(\frac{W}{s}\right), \qquad s = \frac{\max(|W|)}{127}$$

The **de-quantized** approximation is $\hat{W} = s \cdot W_q$.

The per-element quantization error satisfies $|W - \hat{W}| \leq s/2$.

**Signal-to-Quantization-Noise Ratio (SQNR)**:

$$\text{SQNR} = 10 \log_{10}\frac{\|W\|_F^2}{\|W - \hat{W}\|_F^2} \quad \text{(dB)}$$

Higher SQNR means quantization introduces less relative error.
For INT8 with symmetric uniform quantization, the theoretical SQNR is approximately $6.02 \times 8 = 48$ dB
(6.02 dB per bit).


In [ ]:
def fake_quantize_int8(W: torch.Tensor):
    """Symmetric per-tensor INT8 fake quantization."""
    s = W.abs().max() / 127.0
    W_q = torch.clamp(torch.round(W / s), -127, 127)  # INT8 range
    W_hat = s * W_q  # dequantized
    return W_q.to(torch.int8), W_hat, s

def sqnr(W: torch.Tensor, W_hat: torch.Tensor) -> float:
    signal_power = W.pow(2).sum()
    noise_power = (W - W_hat).pow(2).sum()
    return (10 * torch.log10(signal_power / noise_power)).item()

torch.manual_seed(0)
W = torch.randn(64, 64)  # simulate a weight matrix

W_q, W_hat, scale = fake_quantize_int8(W)
error = (W - W_hat).abs()

print(f"Scale factor s         : {scale:.6f}")
print(f"Max quantization error : {error.max().item():.6f}  (bound: s/2 = {scale/2:.6f})")
print(f"Mean absolute error    : {error.mean().item():.6f}")
print(f"SQNR                   : {sqnr(W, W_hat):.2f} dB")

print()

# Compare FP32 vs INT8 (de-quantized) matmul output
x = torch.randn(16, 64)
out_fp32 = x @ W.T
out_int8 = x @ W_hat.T
rel_error = ((out_fp32 - out_int8).norm() / out_fp32.norm()).item()
print(f"Matmul relative error FP32 vs INT8-dequant: {rel_error:.4f}")

print()

# Show INT8 memory saving
fp32_bytes = W.element_size() * W.numel()
int8_bytes = W_q.element_size() * W_q.numel()
print(f"FP32 size: {fp32_bytes} bytes,  INT8 size: {int8_bytes} bytes,  "
      f"compression ratio: {fp32_bytes/int8_bytes:.1f}x")


## 7.8 Kahan Compensated Summation

Naive floating-point summation of $n$ values accumulates rounding error of order $O(n\epsilon)$,
which can be significant for $n = 10^6$ or more.

The **Kahan summation algorithm** reduces this to $O(\epsilon)$ regardless of $n$ by tracking
a running compensation term $c$ that captures the low-order bits lost at each step:

```
s = 0;  c = 0
for each x_i:
    y = x_i - c           # compensated value
    t = s + y             # new running sum (low bits of y lost here)
    c = (t - s) - y       # recover the lost low bits
    s = t
```

This is critical in LLM training for **gradient accumulation** over many micro-batches,
where the accumulated gradient sum must be as accurate as possible.


In [ ]:
def naive_sum(values: torch.Tensor) -> float:
    """Simple left-to-right summation in FP32."""
    s = 0.0
    for v in values:
        s += v.item()
    return s

def kahan_sum(values: torch.Tensor) -> float:
    """Kahan compensated summation in FP32."""
    s = 0.0
    c = 0.0  # compensation
    for v in values:
        y = v.item() - c
        t = s + y
        c = (t - s) - y
        s = t
    return s

torch.manual_seed(1)
n = 1_000_000
values = torch.rand(n, dtype=torch.float32)  # values in [0, 1)

# Ground truth: sum in FP64
true_sum = values.to(torch.float64).sum().item()

naive = naive_sum(values)
kahan = kahan_sum(values)
torch_sum = values.sum().item()  # PyTorch uses pairwise summation internally

print(f"n = {n:,}")
print(f"True sum (FP64) : {true_sum:.10f}")
print(f"Naive sum (FP32): {naive:.10f}   error = {abs(naive - true_sum):.2e}")
print(f"Kahan sum (FP32): {kahan:.10f}   error = {abs(kahan - true_sum):.2e}")
print(f"torch.sum (FP32): {torch_sum:.10f}   error = {abs(torch_sum - true_sum):.2e}")

# Adversarial case: sum of 1.0 followed by many tiny values
adversarial = torch.cat([torch.tensor([1.0]), torch.full((n - 1,), 1e-7)])
true_adv = adversarial.to(torch.float64).sum().item()
print(f"\nAdversarial case (1.0 + 1e7 * 1e-7):")
print(f"  True     : {true_adv:.6f}")
print(f"  Naive    : {naive_sum(adversarial):.6f}")
print(f"  Kahan    : {kahan_sum(adversarial):.6f}")


## 7.9 Mixed-Precision Training Best Practices

Automatic Mixed Precision (AMP) uses different dtypes for different operations to balance speed and stability:

| Operation                  | Recommended dtype | Reason                                              |
|----------------------------|-------------------|-----------------------------------------------------|
| Forward activations        | BF16 / FP16       | Fast; BF16 safer due to wider exponent range        |
| Attention weights          | BF16              | Good enough for softmax, saves memory               |
| LayerNorm / softmax compute| FP32              | Sensitive to precision; renormalized outputs        |
| Loss accumulation          | FP32              | Tiny values; underflows in FP16                     |
| Embedding lookup           | FP32              | Index operations need precise integer alignment     |
| Optimizer states (Adam)    | FP32              | Momentum and variance accumulation need precision   |
| Master weight copy         | FP32              | Small gradient updates lost in BF16                 |

**Master weight pattern**: keep a FP32 master copy of parameters, cast to BF16 for the forward/backward
pass, then update the FP32 master copy.  This ensures that small but consistent gradient updates
(e.g., $|\Delta w| \approx 10^{-6}$) are not lost to rounding.


In [ ]:
class MixedPrecisionLinear(nn.Module):
    """
    Demonstrates the master-weight pattern:
    - master_weight in FP32 (for accurate gradient accumulation)
    - forward uses BF16 cast for speed
    - gradient applied to FP32 master
    """

    def __init__(self, in_features: int, out_features: int):
        super().__init__()
        # Master copy in FP32
        self.master_weight = nn.Parameter(torch.randn(out_features, in_features))
        self.master_bias = nn.Parameter(torch.zeros(out_features))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Cast to BF16 for compute; result returned in BF16
        w_bf16 = self.master_weight.to(torch.bfloat16)
        b_bf16 = self.master_bias.to(torch.bfloat16)
        return F.linear(x.to(torch.bfloat16), w_bf16, b_bf16)

torch.manual_seed(0)
layer = MixedPrecisionLinear(32, 16)
x = torch.randn(8, 32)

# Forward pass produces BF16 output
out = layer(x)
print(f"Output dtype : {out.dtype}")
print(f"Master weight dtype: {layer.master_weight.dtype}")

# Simulate a tiny gradient update that would be lost in BF16
small_grad = torch.full_like(layer.master_weight, 1e-6)
w_fp32 = layer.master_weight.data.clone()
w_bf16 = w_fp32.to(torch.bfloat16)

# Update in FP32 vs BF16
w_fp32_updated = w_fp32 - 0.01 * small_grad
w_bf16_updated = w_bf16 - (0.01 * small_grad).to(torch.bfloat16)

# How much of the update survived?
fp32_change = (w_fp32_updated - w_fp32).abs().mean().item()
bf16_change = (w_bf16_updated.float() - w_bf16.float()).abs().mean().item()
print(f"\nSmall update (lr=0.01, grad=1e-6):")
print(f"  FP32 master — mean |Δw|: {fp32_change:.2e}")
print(f"  BF16 only   — mean |Δw|: {bf16_change:.2e}  "
      f"({'update lost!' if bf16_change == 0.0 else 'update survived'})")

print()
print("Mixed-precision dtype summary:")
roles = [
    ("Loss accumulation", "FP32"),
    ("Attention weights", "BF16"),
    ("Embedding lookup", "FP32"),
    ("Optimizer states", "FP32"),
    ("Forward activations", "BF16"),
    ("Master weight copy", "FP32"),
]
for role, dtype in roles:
    print(f"  {role:<25} -> {dtype}")


## 7.10 Summary

| Topic                        | Key Takeaway                                                      |
|------------------------------|-------------------------------------------------------------------|
| IEEE 754 formats             | BF16 = FP32 range + half precision; FP16 overflows at 65504      |
| Catastrophic cancellation    | Use Welford for variance; never subtract nearly equal numbers     |
| Condition numbers            | High $\kappa$ amplifies errors; $1/\sqrt{d_k}$ scaling helps    |
| Stable softmax               | Shift by max; log-softmax avoids intermediate overflow            |
| Stable LayerNorm             | Welford variance + $\epsilon$ denominator regularisation         |
| Online softmax               | Maintains $(m, \ell)$ per row; exact result without $N^2$ matrix |
| INT8 quantization            | Scale + round + clamp; SQNR $\approx 48$ dB for 8-bit           |
| Kahan summation              | $O(\epsilon)$ error vs $O(n\epsilon)$ for naive; use in grad acc |
| Mixed precision              | BF16 forward + FP32 optimizer states + master weights            |

Numerical stability is not an afterthought in LLM engineering — it is a fundamental constraint
that shapes architecture choices (attention scaling, LayerNorm placement, residual connections)
and training recipes (loss scaling, gradient clipping, mixed precision).
